# 🛡️ Content Moderation API

**Violence + NSFW Detection** using ViT models

---

⚠️ **Chạy từ Cell 1 → 8, mỗi cell xong mới chạy cell kế tiếp.**

## Cell 1: Bật GPU
`Runtime → Change runtime type → T4 GPU → Save`

In [ ]:
import torch

print(f"CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Cell 2: Cài thư viện

In [ ]:
!pip install -q timm transformers safetensors
!pip install -q fastapi uvicorn python-multipart pyngrok
!pip install -q Pillow

# Kiểm tra ffmpeg
!which ffmpeg && ffmpeg -version | head -1

print("\nCài đặt xong ✅")

## Cell 3: Load 2 Models

In [ ]:
import torch
import timm
import json
from PIL import Image
from torchvision import transforms
from transformers import ViTImageProcessor, ViTForImageClassification
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ═══ MODEL 1: VIOLENCE ═══
print("Loading Violence model...")
vit_violence = timm.create_model("vit_base_patch16_224", pretrained=False, num_classes=2)
weight_path = hf_hub_download(
    repo_id="jaranohaal/vit-base-violence-detection",
    filename="model.safetensors"
)
state_dict = load_file(weight_path)
vit_violence.load_state_dict(state_dict, strict=True)
vit_violence.to(device)
vit_violence.eval()

violence_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])
print("Violence model loaded ✅")

# ═══ MODEL 2: NSFW (AdamCodd) ═══
print("\nLoading NSFW model...")
nsfw_processor = ViTImageProcessor.from_pretrained("AdamCodd/vit-base-nsfw-detector")
nsfw_model = ViTForImageClassification.from_pretrained("AdamCodd/vit-base-nsfw-detector")
nsfw_model.to(device)
nsfw_model.eval()
print(f"NSFW labels: {nsfw_model.config.id2label}")
print("NSFW model loaded ✅")

# ═══ VRAM ═══
print(f"\nVRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB used")

## Cell 4: Config + Tất cả hàm (dùng ffmpeg cho video)

In [ ]:
import os
import glob
import subprocess
import shutil
import numpy as np

# ═══ CONFIG ═══
VIOLENCE_THRESHOLD = 0.7
NSFW_THRESHOLD = 0.7
VIDEO_SENSITIVE_RATIO = 0.10
# Trích 1 frame mỗi N giây (thay vì mỗi N frame)
FRAME_EXTRACT_INTERVAL = 1  # 1 frame / giây

UPLOAD_DIR = "/tmp/moderation_uploads"
os.makedirs(UPLOAD_DIR, exist_ok=True)


# ═══ FFPROBE: lấy thông tin video ═══
def get_video_info(video_path):
    """Dùng ffprobe lấy fps, duration, total_frames"""
    cmd = [
        "ffprobe", "-v", "quiet",
        "-print_format", "json",
        "-show_format", "-show_streams",
        video_path
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"[FFPROBE ERROR] {result.stderr[:300]}")
        return None

    import json as _json
    info = _json.loads(result.stdout)

    # Tìm video stream
    for stream in info.get("streams", []):
        if stream.get("codec_type") == "video":
            fps_str = stream.get("r_frame_rate", "30/1")
            parts = fps_str.split("/")
            fps = float(parts[0]) / float(parts[1]) if len(parts) == 2 and float(parts[1]) != 0 else 30.0

            duration = float(stream.get("duration", 0))
            if duration == 0:
                duration = float(info.get("format", {}).get("duration", 0))

            nb_frames = int(stream.get("nb_frames", 0))
            if nb_frames == 0:
                nb_frames = int(fps * duration)

            return {
                "fps": round(fps, 2),
                "duration": round(duration, 2),
                "total_frames": nb_frames,
                "width": int(stream.get("width", 0)),
                "height": int(stream.get("height", 0)),
                "codec": stream.get("codec_name", "unknown")
            }
    return None


# ═══ FFMPEG: trích frames ra ảnh ═══
def extract_frames_ffmpeg(video_path, interval=1):
    """
    Dùng ffmpeg trích 1 frame mỗi `interval` giây.
    Trả về list PIL Images.
    """
    frames_dir = video_path + "_frames"
    os.makedirs(frames_dir, exist_ok=True)

    cmd = [
        "ffmpeg", "-y",
        "-i", video_path,
        "-vf", f"fps=1/{interval}",   # 1 frame mỗi N giây
        "-q:v", "2",                    # chất lượng JPEG cao
        "-f", "image2",
        os.path.join(frames_dir, "frame_%05d.jpg")
    ]

    print(f"[FFMPEG] Extracting frames (1 per {interval}s)...")
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=600)

    if result.returncode != 0:
        print(f"[FFMPEG ERROR] {result.stderr[-500:]}")
        shutil.rmtree(frames_dir, ignore_errors=True)
        return []

    # Load tất cả frame images
    frame_files = sorted(glob.glob(os.path.join(frames_dir, "frame_*.jpg")))
    print(f"[FFMPEG] Extracted {len(frame_files)} frames")

    frames = []
    for f in frame_files:
        try:
            img = Image.open(f).convert("RGB")
            frames.append(img)
        except Exception as e:
            print(f"[WARN] Skip frame {f}: {e}")

    # Dọn dẹp
    shutil.rmtree(frames_dir, ignore_errors=True)

    return frames


# ═══ DETECT VIOLENCE ═══
def detect_violence(image):
    if not isinstance(image, Image.Image):
        image = Image.fromarray(image).convert("RGB")
    else:
        image = image.convert("RGB")

    input_tensor = violence_transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = vit_violence(input_tensor)

    probs = torch.softmax(outputs, dim=1)
    violence_score = probs[0][1].item()

    return {
        "is_violence": violence_score > VIOLENCE_THRESHOLD,
        "score": round(violence_score, 4)
    }


# ═══ DETECT NSFW ═══
def detect_nsfw(image):
    if not isinstance(image, Image.Image):
        image = Image.fromarray(image).convert("RGB")
    else:
        image = image.convert("RGB")

    inputs = nsfw_processor(images=image, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = nsfw_model(**inputs)

    probs = torch.softmax(outputs.logits, dim=1)
    labels = nsfw_model.config.id2label

    nsfw_idx = None
    for idx, label in labels.items():
        if "nsfw" in label.lower():
            nsfw_idx = int(idx)
            break

    nsfw_score = probs[0][nsfw_idx].item() if nsfw_idx is not None else 0.0

    all_scores = {}
    for idx, label in labels.items():
        all_scores[label] = round(probs[0][int(idx)].item(), 4)

    return {
        "is_nsfw": nsfw_score > NSFW_THRESHOLD,
        "score": round(nsfw_score, 4),
        "all_scores": all_scores
    }


# ═══ MODERATE IMAGE ═══
def moderate_image(image_path):
    image = Image.open(image_path).convert("RGB")

    violence_result = detect_violence(image)

    if violence_result["is_violence"]:
        return {
            "is_sensitive": True,
            "categories": ["violence"],
            "violence": violence_result,
            "nsfw": {"is_nsfw": False, "skipped": True}
        }

    nsfw_result = detect_nsfw(image)
    categories = ["nsfw"] if nsfw_result["is_nsfw"] else []

    return {
        "is_sensitive": len(categories) > 0,
        "categories": categories,
        "violence": violence_result,
        "nsfw": nsfw_result
    }


# ═══ MODERATE VIDEO (100% ffmpeg, không dùng OpenCV) ═══
def moderate_video(video_path):
    file_size = os.path.getsize(video_path)
    print(f"\n{'='*50}")
    print(f"[VIDEO] File: {video_path}")
    print(f"[VIDEO] Size: {file_size / 1024 / 1024:.1f} MB")

    # 1. Lấy thông tin video bằng ffprobe
    video_info = get_video_info(video_path)
    if video_info is None:
        return {
            "is_sensitive": False,
            "categories": [],
            "error": "ffprobe cannot read this video file.",
            "details": {
                "total_frames": 0, "sampled_frames": 0, "fps": 0,
                "violence": {"sensitive_count": 0, "ratio": 0, "detected": False},
                "nsfw": {"sensitive_count": 0, "ratio": 0, "detected": False},
                "frames_skipped_nsfw": 0
            }
        }

    print(f"[VIDEO] Info: {video_info}")

    # 2. Trích frames bằng ffmpeg
    frames = extract_frames_ffmpeg(video_path, interval=FRAME_EXTRACT_INTERVAL)

    if len(frames) == 0:
        return {
            "is_sensitive": False,
            "categories": [],
            "error": "ffmpeg could not extract any frames.",
            "details": {
                "total_frames": video_info["total_frames"],
                "sampled_frames": 0, "fps": video_info["fps"],
                "violence": {"sensitive_count": 0, "ratio": 0, "detected": False},
                "nsfw": {"sensitive_count": 0, "ratio": 0, "detected": False},
                "frames_skipped_nsfw": 0
            }
        }

    num_sampled = len(frames)
    print(f"[VIDEO] Processing {num_sampled} frames...")

    # 3. Chạy detection trên từng frame
    violence_count = 0
    nsfw_count = 0
    skipped_count = 0

    for i, frame in enumerate(frames):
        if i % 5 == 0:
            print(f"[VIDEO] Frame {i+1}/{num_sampled}")

        v_result = detect_violence(frame)

        if v_result["is_violence"]:
            violence_count += 1
            skipped_count += 1
            continue

        n_result = detect_nsfw(frame)
        if n_result["is_nsfw"]:
            nsfw_count += 1

    violence_ratio = violence_count / num_sampled if num_sampled > 0 else 0
    nsfw_ratio = nsfw_count / num_sampled if num_sampled > 0 else 0

    categories = []
    if violence_ratio > VIDEO_SENSITIVE_RATIO:
        categories.append("violence")
    if nsfw_ratio > VIDEO_SENSITIVE_RATIO:
        categories.append("nsfw")

    print(f"[VIDEO] ✅ Done! Violence: {violence_count}/{num_sampled}, NSFW: {nsfw_count}/{num_sampled}")

    return {
        "is_sensitive": len(categories) > 0,
        "categories": categories,
        "details": {
            "total_frames": video_info["total_frames"],
            "sampled_frames": num_sampled,
            "fps": video_info["fps"],
            "duration": video_info["duration"],
            "codec": video_info["codec"],
            "resolution": f"{video_info['width']}x{video_info['height']}",
            "violence": {
                "sensitive_count": violence_count,
                "ratio": round(violence_ratio, 4),
                "detected": violence_ratio > VIDEO_SENSITIVE_RATIO
            },
            "nsfw": {
                "sensitive_count": nsfw_count,
                "ratio": round(nsfw_ratio, 4),
                "detected": nsfw_ratio > VIDEO_SENSITIVE_RATIO
            },
            "frames_skipped_nsfw": skipped_count
        }
    }

print("Tất cả hàm OK ✅")

## Cell 5: Test ảnh

In [ ]:
!wget -q -O test_safe.jpg "https://picsum.photos/640/480"

image = Image.open("test_safe.jpg").convert("RGB")

print("=== Violence ===")
v = detect_violence(image)
print(json.dumps(v, indent=2))

print("\n=== NSFW ===")
n = detect_nsfw(image)
print(json.dumps(n, indent=2))

print("\n=== Full Pipeline ===")
result = moderate_image("test_safe.jpg")
print(json.dumps(result, indent=2))

if result["is_sensitive"]:
    print(f"\n⚠️ NHẠY CẢM: {result['categories']}")
else:
    print(f"\n✅ AN TOÀN")

## Cell 5b: Test video trực tiếp (download sample)

In [ ]:
# Download 1 video mẫu nhỏ để test
!wget -q -O test_video.mp4 "https://sample-videos.com/video321/mp4/240/big_buck_bunny_240p_1mb.mp4"

print(f"File size: {os.path.getsize('test_video.mp4')} bytes")

# Test ffprobe
info = get_video_info("test_video.mp4")
print(f"\nVideo info: {json.dumps(info, indent=2)}")

# Test full pipeline
result = moderate_video("test_video.mp4")
print(f"\nResult: {json.dumps(result, indent=2)}")

## Cell 6: FastAPI Server

In [ ]:
from pathlib import Path
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.middleware.cors import CORSMiddleware
import uvicorn
import threading

app = FastAPI(title="Content Moderation API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/health")
def health():
    return {"status": "ok", "device": str(device)}

@app.post("/moderate/image")
async def api_moderate_image(file: UploadFile = File(...)):
    temp_path = os.path.join(UPLOAD_DIR, f"img_{file.filename}")
    content = await file.read()
    with open(temp_path, "wb") as f:
        f.write(content)
        f.flush()
        os.fsync(f.fileno())
    print(f"[API] Image: {file.filename}, {len(content)} bytes")
    try:
        result = moderate_image(temp_path)
    finally:
        if os.path.exists(temp_path):
            os.remove(temp_path)
    return result

@app.post("/moderate/video")
async def api_moderate_video(
    file: UploadFile = File(...),
    user_id: str = Form(default="anonymous")
):
    ext = Path(file.filename).suffix.lower() or '.mp4'
    temp_path = os.path.join(UPLOAD_DIR, f"vid_upload{ext}")

    # Ghi file theo chunks
    total_bytes = 0
    with open(temp_path, "wb") as f:
        while True:
            chunk = await file.read(1024 * 1024)  # 1MB
            if not chunk:
                break
            f.write(chunk)
            total_bytes += len(chunk)
        f.flush()
        os.fsync(f.fileno())

    print(f"[API] Video: {file.filename}, {total_bytes} bytes written to {temp_path}")

    disk_size = os.path.getsize(temp_path)
    if disk_size < 1000:
        os.remove(temp_path)
        return {"error": f"File too small ({disk_size} bytes)", "is_sensitive": False, "categories": []}

    try:
        result = moderate_video(temp_path)
    finally:
        # Cleanup
        for p in [temp_path]:
            if os.path.exists(p):
                os.remove(p)
        frames_dir = temp_path + "_frames"
        if os.path.exists(frames_dir):
            shutil.rmtree(frames_dir, ignore_errors=True)

    return result

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

import time
time.sleep(2)
print("Server OK ✅ → http://localhost:8000")

## Cell 7: Ngrok
⚠️ **Thay token ngrok của bạn**

In [ ]:
from pyngrok import ngrok

NGROK_TOKEN = "PASTE_TOKEN_CUA_BAN"

ngrok.set_auth_token(NGROK_TOKEN)
tunnel = ngrok.connect(8000)

url_str = str(tunnel)
start = url_str.index("https://")
end = url_str.index('"', start + 1) if '"' in url_str[start + 1:] else len(url_str)
BASE_URL = url_str[start:end]

print(f"🌐 API: {BASE_URL}")
print(f"📄 Docs: {BASE_URL}/docs")

## Cell 8: Test API

In [ ]:
import requests

HEADERS = {"ngrok-skip-browser-warning": "true"}

# Health
print("=== Health ===")
r = requests.get(f"{BASE_URL}/health", headers=HEADERS)
print(r.json())

# Image
print("\n=== Image ===")
with open("test_safe.jpg", "rb") as f:
    r = requests.post(
        f"{BASE_URL}/moderate/image",
        files={"file": ("test.jpg", f, "image/jpeg")},
        headers=HEADERS
    )
print(json.dumps(r.json(), indent=2))

# Video
print("\n=== Video ===")
if os.path.exists("test_video.mp4"):
    with open("test_video.mp4", "rb") as f:
        r = requests.post(
            f"{BASE_URL}/moderate/video",
            files={"file": ("test.mp4", f, "video/mp4")},
            data={"user_id": "test"},
            headers=HEADERS
        )
    print(json.dumps(r.json(), indent=2))
else:
    print("No test_video.mp4 found, skip video test")